In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# models and metrics
import xgboost as xgb
from sklearn.model_selection import train_test_split
from xgbse.metrics import concordance_index
from xgbse.non_parametric import get_time_bins
from xgbse import (
    XGBSEKaplanNeighbors,
    XGBSEKaplanTree,
    XGBSEDebiasedBCE,
    XGBSEBootstrapEstimator,
)
from xgbse.converters import convert_data_to_xgb_format, convert_to_structured
import optuna
import shap
from lifelines import KaplanMeierFitter

import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "Arial",   # 或 "Times New Roman"
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "legend.fontsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "pdf.fonttype": 42,       # 重要！避免字体变成 Type 3
    "ps.fonttype": 42,
    "figure.dpi": 300,
    "savefig.dpi": 600
})

In [ ]:
x = pd.read_csv(r'E:\UKB\data\wxx-lung cancer\x_train_lung.csv')
y = pd.read_csv(r'E:\UKB\data\wxx-lung cancer\y_train_lung.csv')
x2 = pd.read_csv(r'E:\UKB\data\wxx-lung cancer\x_test_lung.csv')
y2 = pd.read_csv(r'E:\UKB\data\wxx-lung cancer\y_test_lung.csv')
x = x.drop(["Unnamed: 0"], axis=1)
x2 = x2.drop(["Unnamed: 0"], axis=1)
y = y.drop(["Unnamed: 0"], axis=1)
y2 = y2.drop(["Unnamed: 0"], axis=1)
y = convert_to_structured(y["time"], y["LC"])
y2 = convert_to_structured(y2["time"], y2["LC"])
dtrain = convert_data_to_xgb_format(x, y, "survival:cox")
dval = convert_data_to_xgb_format(x2, y2, "survival:cox")

In [ ]:
def objective(trial):
    PARAMS_XGB_COX = {
    "objective": "survival:cox",
    "eval_metric": "cox-nloglik",
    "tree_method": "hist",
    "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-1),
    "max_depth": trial.suggest_int("max_depth", 3, 10),
    "booster": "dart",
    "subsample": trial.suggest_float("subsample", 0.5, 0.9),
    "min_child_weight": trial.suggest_int("min_child_weight", 10, 30),
    "colsample_bynode": trial.suggest_float("colsample_bynode", 0.5, 0.9),
    "device":"cuda",
    "seed":42
}
    
    bst = xgb.cv(
        PARAMS_XGB_COX,
        dtrain,
        num_boost_round=2000,
        early_stopping_rounds=30,
        nfold=5,
        verbose_eval=0,
    )
    
    best_iteration = bst['test-cox-nloglik-mean'].min()
    return best_iteration 
study = optuna.create_study(direction="minimize")

study.optimize(objective, n_trials=200)

In [ ]:
params = {
    "objective": "survival:cox",
    "eval_metric": "cox-nloglik",
    "tree_method": "hist",
    'learning_rate': 0.08028460584543722,
    'max_depth': 3,
    'subsample': 0.6573366724793972,
    'min_child_weight': 18,
    'colsample_bynode': 0.6745398065697549,
    'seed':2024
}

bst = xgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    early_stopping_rounds=30,
    evals=[(dval, "val")],
    verbose_eval=0,
)

explainer = shap.TreeExplainer(bst)
shap_values = explainer.shap_values(x)

feature_names = x.columns.tolist()

mean_shap_values = np.abs(shap_values).mean(axis=0)
sorted_indices = np.argsort(mean_shap_values)[::-1]  

selected_features = []
c_indexes = []

for idx in sorted_indices:

    current_feature = feature_names[idx]
    
    selected_features.append(current_feature)
    
    dtrain_selected = convert_data_to_xgb_format(x[selected_features], y, "survival:cox")
    dval_selected = convert_data_to_xgb_format(x2[selected_features], y2, "survival:cox")

    bst_selected = xgb.train(
        params,
        dtrain_selected,
        num_boost_round=2000,
        early_stopping_rounds=30,
        evals=[(dval_selected, "val")],
        verbose_eval=0,
    )
    
    preds_selected = bst_selected.predict(dval_selected)
    cind = concordance_index(y2, preds_selected, risk_strategy="precomputed")
    c_indexes.append(cind)

    print(f"Selected Features: {selected_features}, C-index: {cind:.3f}")

print("Final Selected Features:", selected_features)
print("C-index Values:", c_indexes)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

mean_shap_values = np.abs(shap_values).mean(axis=0)

shap_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean SHAP Value': mean_shap_values
})

shap_df = shap_df.sort_values(by='Mean SHAP Value', ascending=False)

plt.figure(figsize=(12, 6))

# C-index 
sns.lineplot(x=selected_features, y=c_indexes, marker='o', label='C-index', color='b')
plt.xticks(rotation=90)  
plt.title('Sequential Forward Selection from Preselected Candidate Proteins')
plt.xlabel('Selected Features')
plt.ylabel('C-index')

plt.grid()

ax2 = plt.gca().twinx()
sns.barplot(x=shap_df['Feature'], y=shap_df['Mean SHAP Value'], ax=ax2, alpha=0.3, color='orange', label='Mean SHAP Value')
ax2.set_ylabel('Mean SHAP Value')

plt.legend(loc='upper left')
plt.tight_layout()  
plt.savefig('E:/UKB/data/wxx-lung cancer/sequential_feature_selection.png',
    dpi=600)
plt.show()

In [ ]:
selected_features = ['CXCL17', 'TNR', 'WFDC2', 'GDF15', 'EDA2R', 'ALPP', 'CDCP1', 'SFTPD', 'CLEC3B', 'MMP12', 'CEACAM5', 'OLR1', 'SPINT1', 'IL22', 'ALDH3A1']

In [ ]:
dtrain_selected = convert_data_to_xgb_format(x[selected_features], y, "survival:cox")
def objective(trial):
    
    PARAMS_XGB_COX = {
    "objective": "survival:cox",
    "eval_metric": "cox-nloglik",
    "tree_method": "hist",
    "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-1),
    "max_depth": trial.suggest_int("max_depth", 3, 10),
    "booster": "dart",
    "subsample": trial.suggest_float("subsample", 0.5, 0.9),
    "min_child_weight": trial.suggest_int("min_child_weight", 10, 30),
    "colsample_bynode": trial.suggest_float("colsample_bynode", 0.5, 0.9),
    "device":"cuda",
    "seed":42
}
    
    bst = xgb.cv(
        PARAMS_XGB_COX,
        dtrain_selected,
        num_boost_round=2000,
        early_stopping_rounds=30,
        nfold=5,
        verbose_eval=0,
    )
    
    best_iteration = bst['test-cox-nloglik-mean'].min()
    return best_iteration 
study = optuna.create_study(direction="minimize")

study.optimize(objective, n_trials=200)

In [ ]:
dval_selected = convert_data_to_xgb_format(x2[selected_features], y2, "survival:cox")

params = {
    "objective": "survival:cox",
    "eval_metric": "cox-nloglik",
    "tree_method": "hist",
    'learning_rate': 0.08028460584543722,
    'max_depth': 3,
    'subsample': 0.6573366724793972,
    'min_child_weight': 18,
    'colsample_bynode': 0.6745398065697549,
    'seed':2024
}

bst_selected = xgb.train(
        params,
        dtrain_selected,
        num_boost_round=2000,
        early_stopping_rounds=30,
        evals=[(dval_selected, "val")],
        verbose_eval=0,
)
    
preds_selected = bst_selected.predict(dval_selected)
cind = concordance_index(y2, preds_selected, risk_strategy="precomputed")
c_indexes.append(cind)

print(f"Selected Features: {selected_features}, C-index: {cind:.3f}")

In [ ]:
x3=x2[selected_features]
import shap
shap_values = shap.TreeExplainer(bst_selected).shap_values(x3)
# ===== 1. SHAP dot summary plot =====
plt.figure()
shap.summary_plot(shap_values, x3, show=False)

plt.savefig(
    'E:/UKB/data/wxx-lung cancer/shap_summary.png',
    dpi=600,
    bbox_inches='tight'
)
plt.close()


# ===== 2. SHAP bar plot =====
plt.figure()
shap.summary_plot(
    shap_values,
    x3,
    plot_type="bar",
    show=False
)

plt.savefig(
    'E:/UKB/data/wxx-lung cancer/shap_bar.png',
    dpi=600,
    bbox_inches='tight'
)
plt.close()

In [ ]:
explainer = shap.Explainer(bst_selected, x3)
shap_values = explainer(x3)
shap.plots.bar(shap_values,max_display=15,show=False)

In [ ]:
shap.plots.scatter(shap_values[:, "CXCL17"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/CXCL17.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "GDF15"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/GDF15.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "EDA2R"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/EDA2R.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "TNR"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/TNR.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "CDCP1"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/CDCP1.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "ALPP"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/ALPP.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "CLEC3B"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/CLEC3B.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "CEACAM5"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/CEACAM5.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "SFTPD"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/SFTPD.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "WFDC2"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/WFDC2.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "SPINT1"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/SPINT1.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "ALDH3A1"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/ALDH3A1.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "IL22"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/IL22.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "OLR1"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/OLR1.png',
    dpi=600, bbox_inches='tight')
shap.plots.scatter(shap_values[:, "MMP12"],show=False)
plt.savefig('E:/UKB/data/wxx-lung cancer/MMP12.png',
    dpi=600, bbox_inches='tight')

In [ ]:
y = pd.read_csv(r'E:\UKB\data\wxx-lung cancer\y_train_lung.csv')
y2 = pd.read_csv(r'E:\UKB\data\wxx-lung cancer\y_test_lung.csv')

y = y.drop(["Unnamed: 0"], axis=1)
y2 = y2.drop(["Unnamed: 0"], axis=1)
y= y.drop(["f.eid"], axis=1)
y2 = y2.drop(["f.eid"], axis=1)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from lifelines import KaplanMeierFitter

T = y2['time']
E = y2['CD']
X = x2

survival_probs = bst_selected.predict(dval_selected)  

kmf = KaplanMeierFitter()
kmf.fit(T, event_observed=E)

time_points = 10
survival_probs_10_years=kmf.predict(time_points)

plt.figure(figsize=(8, 8))

fpr, tpr, _ = roc_curve(E, survival_probs)
    
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, lw=2)

plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.ylabel('Sensitivity')
plt.xlabel('1 - Specificity')
plt.title('ROC Curve of Plasma Proteins on Predicting Lung Cancer Risk')
plt.savefig('E:/UKB/data/wxx-lung cancer/roc.png',dpi=600, bbox_inches='tight')
plt.grid()
plt.show()
roc_auc

In [ ]:
y = convert_to_structured(y["time"], y["LC"])
y2 = convert_to_structured(y2["time"], y2["LC"])

In [ ]:
from sklearn.metrics import roc_auc_score
from lifelines.utils import concordance_index
from sksurv.metrics import (
    concordance_index_censored,
    concordance_index_ipcw,
    cumulative_dynamic_auc,
    integrated_brier_score,
)

times= np.arange(1,14,1)
risk_scores = bst_selected.predict(dval_selected)  
xgb_auc,xgb_mean_auc=cumulative_dynamic_auc(y, y2, risk_scores, times)
plt.plot(times, xgb_auc, marker="o", label='AUC of each year')
plt.axhline(xgb_mean_auc, linestyle="--")
plt.xlabel("years from enrollment")
plt.ylabel("time-dependent AUC")
plt.legend(loc='upper right')
plt.savefig('E:/UKB/data/wxx-lung cancer/time_dependent_auc.png',dpi=600, bbox_inches='tight')

plt.grid(True)
xgb_auc

In [ ]:
shap_values_total = np.sum(shap_values.values,axis=1)
shap_values_total=shap_values_total+shap_values.base_values
shap_values_total=pd.DataFrame(shap_values_total)
x4=x2[selected_features]
explainer = shap.Explainer(bst_selected, x4)
shap_values2 = explainer(x4)
shap_values_total2 = np.sum(shap_values2.values,axis=1)
shap_values_total2=shap_values_total2+shap_values2.base_values
shap_values_total2=pd.DataFrame(shap_values_total2)

In [ ]:
shap_values_total.to_csv('shap_train_lung.csv', index=False)
shap_values_total2.to_csv('shap_test_lung.csv', index=False)